# Chapter 27: Case Study — Coastal Dynamics

*Part V — Domain Modeling: Land Use & Coastal Systems*

Implemented by the [`brmangue-dissmodel`](https://github.com/DisSModel/brmangue-dissmodel) package (version 0.5.0, installed from GitHub).

<div class="admonition warning">
<p class="admonition-title">Work in progress</p>
<p>The package itself is work in progress: its comparison with TerraME covers two scenarios so far, and its authors ask that it not yet be relied upon for scientific conclusions. The numbers below are those its repository reports for those two scenarios. No runtime figure is given here: the raster–vector runtime comparison depends on the machine, and the package's benchmark executor measures it on your own data.</p>
</div>

## Learning Objectives

By the end of this chapter you will be able to:

- Describe the coupled flood-propagation and mangrove-migration model this case study is built on
- Compare its raster and vector implementations, substrate for substrate
- Explain why flood propagation needs a push-based algorithm rather than `CellularAutomaton`'s pull-based `rule(idx)`
- Know the two independent ways this model validates itself

In [ ]:
# Standard imports — add chapter-specific imports below
import numpy as np
import matplotlib.pyplot as plt

Every substrate comparison in this book so far — Chapter 24's `GameOfLife` and `FireModel` — used a toy or synthetic case to make the vector-versus-raster point. This chapter closes Part V with the ecosystem's deepest real one: `brmangue-dissmodel`, a coupled model of coastal flooding and mangrove migration, implemented on both substrates deliberately, then checked against each other and against the original TerraME reference at once.

## The BR-MANGUE Model

`brmangue-dissmodel` re-implements, in Python, the spatially explicit coastal model BR-MANGUE (Bezerra, Amaral & Kampel, 2013), comparing it with the original TerraME version, kept in the repository `brmangue-terrame`. It couples two processes on the same grid:

1. **Flood dynamics** — sea-level rise propagating inland, with terrain elevation adjusting the water's reach cell by cell.
2. **Mangrove migration** — the ecosystem's response to that rising water: soil-state transitions and sediment accretion tracking where the mangrove can and can't persist.

Both processes share one `RasterBackend` (or one `GeoDataFrame`, on the vector substrate), coordinated within a single `Environment` — a `SyncRasterModel`, which keeps two models' reads and writes consistent within one tick.

## Flood Propagation: A Push Model, Not a Pull Rule

Every cellular automaton this book has built since Chapter 24 follows `CellularAutomaton.rule(idx)`'s pull contract: each cell asks its neighbors for their state and decides its own next value. Flood propagation doesn't fit that shape naturally — water doesn't wait for a cell to ask "am I now flooded"; it originates at a source (the rising sea) and pushes outward, cell by cell, as elevation permits.

`brmangue-dissmodel`'s flood model is built directly on `SpatialModel` (or `RasterModel`) with a free-form `execute()`, not on `CellularAutomaton` — exactly the case the substrate classes of Chapter 22 leave room for: `CellularAutomaton`'s stricter `rule(idx)` contract is the right choice when a transition genuinely depends only on a cell's own neighbors, and the wrong one whenever a process spreads *from* a source outward instead. A push-based flood algorithm, faithful to the original TerraME implementation, is this book's clearest example of that second case.

## Two Substrates, Deliberately

`brmangue-dissmodel` ships genuinely parallel implementations, not one canonical version with an afterthought port:

- **Raster** (`brmangue.models.raster`) — NumPy/`RasterBackend`, vectorized, the fast path. This is the version validated directly against TerraME's own golden-output CSVs.
- **Vector** (`brmangue.models.vector`) — `GeoDataFrame`/`libpysal`, cell-by-cell over real polygon geometry. Numerically equivalent to the raster version, confirmed by a dedicated benchmark executor rather than assumed.

Both implement intentionally identical equations, thresholds, parameter names, and update ordering — the substrate is the only thing that differs, exactly the "same conceptual model, two substrates" pattern Chapter 22 first previewed and Chapter 24 measured concretely on `FireModel`.

## Performance: Measure It on Your Data

Chapter 24 measured the vector-versus-raster gap on `FireModel` and showed where it comes from: per-cell Python calls against whole-array operations. BR-MANGUE's two substrates differ in the same way, and the package ships a benchmark executor, `brmangue_benchmark`, that runs both on the same input and reports their runtime per step next to their agreement. Its authors deliberately publish no single speed ratio — the number depends on the machine and on the input — and recommend running the benchmark on your own grid. The raster implementation is the one compared with TerraME, because it is the one meant for study areas of tens of thousands of cells.

## Two Independent Checks

BR-MANGUE is checked two distinct ways — neither alone is sufficient:

- **Raster vs. vector**: the benchmark executor runs both substrates on the same input and reports, per output band, the share of matching cells, the mean absolute error and the largest error. On a synthetic 60 × 60 grid (3,600 cells, 10 steps) the two agree on every cell of the land-use (`uso`) and soil (`solo`) bands, and the elevation (`alt`) agrees with a mean absolute error of 0.0011 m and a largest error of 0.024 m.
- **Raster vs. TerraME**: a validation executor compares the raster output with reference CSVs written by the TerraME version at fixed steps of a run over the Ilha do Maranhão (50,496 cells) — not "close to our own vector version," but "close to the original implementation itself."

The comparison with TerraME has two scenarios so far:

| Scenario | Steps | `uso` | `solo` | `alt` within 1 mm | largest `alt` error |
|---|---|---|---|---|---|
| Baseline, sea-level rise 0.05 m per step | 19 | 100.0% | 100.0% | 97.4% | 0.10 m |
| Flooding, 0.5 m per step (the lab's own parameters) | 11 | 99.99% (4 cells differ) | 100.0% | 94.4% | 1.01 m |

The categorical bands agree almost cell for cell; the continuous one, elevation, does not, and the reason is instructive. The flux rule compares accumulated elevations with `<=`, and about 57% of neighbouring cells hold *exactly* equal elevations. TerraME adds the contributions cell by cell; NumPy adds them direction by direction. A difference of 10⁻¹⁶ in the order of summation flips an occasional comparison, and since flooding is a threshold, the flips grow with the number of steps. Fed TerraME's exact state, the Python rule reproduces TerraME's next step to 10⁻¹⁴. So the right reading is "same dynamics, not bit-identical" — and the right check is a tolerance for the continuous band and exact agreement for the categorical ones, chosen band by band. One more caution comes from the baseline: at 0.05 m per step no cell is ever flooded, in either implementation, so its perfect agreement mostly tests the mangrove component. That is why the second, flooding scenario exists.

## Exercises

1. **Push versus pull, concretely.** Using *Flood Propagation*'s distinction, sketch — in words, no code needed — why a pull-based `rule(idx)` implementation of sea-level rise would need every cell to somehow "know" whether water has reached any upstream neighbor yet, and why that's awkward to express as a purely local rule.
2. **Which substrate for which purpose.** Given *Two Substrates, Deliberately* and *Performance*, when would you reach for the vector implementation despite its performance cost, and when would raster be the only practical choice? Answer using Chapter 24's own substrate-choice guidance, not intuition alone.
3. **Why two validation checks, not one.** Explain, in your own words, what a model could get wrong that a raster-vs-vector benchmark alone would catch but a raster-vs-TerraME comparison alone would miss, or vice versa.
4. **Ties, by hand.** Take three numbers whose exact sum is 0.3 — for example 0.1, 0.2 and 0.0 — and add them in two different orders in Python. Are the two results equal? Use the answer to explain why BR-MANGUE's elevation band cannot be expected to match TerraME bit for bit, while its land-use band nearly does.

In [ ]:
# Your code here

## Summary

### Key concepts introduced

- BR-MANGUE (Bezerra et al., 2013) as the ecosystem's deepest validation case: one coupled flood/mangrove-migration model, two deliberately parallel substrates, two independent checks
- Push-based flood propagation as the clearest example in this book of a process that doesn't fit `CellularAutomaton`'s pull-based `rule(idx)` contract, and why `SpatialModel`'s free-form `execute()` is the right base class for it instead
- The same vector-vs-raster performance gap Chapter 24 measured on `FireModel`, now at the scale a real coastal study domain actually requires
- Two checks, run independently — raster vs. vector and raster vs. TerraME — neither substituting for the other
- Exact agreement for categorical bands and a tolerance for continuous ones, chosen band by band; ties in floating-point comparisons as a real, explainable source of difference; and a scenario that never floods as a reminder that agreement only covers what a scenario exercises

This closes Part V. Part VI picks the story back up from a different angle — not a specific domain model, but what every model in Parts IV and V needs before its results can be trusted: input data prepared on a common grid, a record of every run, and a check against a reference implementation, brought together at the end in one complete experiment.

## Further Reading

- Bezerra, D. da S., Amaral, S., & Kampel, M. (2013). Impactos da elevação do nível médio do mar sobre o ecossistema manguezal: a contribuição do sensoriamento remoto e modelos computacionais. *Ciência e Natura*, 35(2), 152–162. <https://doi.org/10.5902/2179460X12569> — the BR-MANGUE model
- Bezerra, D. da S., Sousa, F. M., & Costa, S. S. (2026). *brmangue-dissmodel: The BR-MANGUE coupled flood and mangrove succession model for DisSModel* (Version v0.5.0) [Computer software]. Zenodo. <https://doi.org/10.5281/zenodo.23240385>
- brmangue-terrame, the TerraME version it is compared with: <https://github.com/LambdaGeo/brmangue-terrame>
- brmangue-qgis, the QGIS client for this model covered in Chapter 33: <https://github.com/DisSModel/brmangue-qgis>